# C03–M2–L2 — Practice Project: Food Recommendation System Using Chroma DB (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original files](https://github.com/hengkuankian/learning_ibm_rag_agentic/tree/main/Course3_Vector_Databases_for_RAG/Food_Recommendation_System_Using_Chroma_DB) · cross-checked with the [Hugging Face reference port](https://github.com/oooookk7/huggingface-rag-and-agentic-ai/tree/main/03-vector-databases-for-rag)

**Before you run:** add **`GOOGLE_API_KEY`** to Colab 🔑 Secrets (free key from aistudio.google.com/apikey). CPU runtime is enough — no GPU, no model downloads.

**What changed vs. the IBM version (CP-G.3 Gemini migration)**
- Setup: Cell 1 = minimal pip install (+ `chromadb gradio`), Cell 2 = universal Gemini init. `ibm-watsonx-ai`, `sentence-transformers`, `get_llm()`, the watsonx shim and `llm_factory.py` removed.
- `llm_factory.py` → `%%writefile gemini_ef.py` (`GeminiEmbeddingFunction`); `shared_functions.py` uses it instead of `SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")`.
- `enhanced_rag_chatbot.py`: watsonx credentials + `ModelInference(ibm/granite-3-3-8b-instruct)` → `ChatGoogleGenerativeAI(model="gemini-3.8-flash")`; `model.generate(...)["results"][0]["generated_text"]` → `model.invoke(prompt).text`. Fallback logic unchanged.
- Other scripts (`interactive_search.py`, `advanced_search.py`, `system_comparison.py`, exercise files) unchanged; `FoodDataSet.json` via `wget`; CLIs run with piped demo answers or `%run`.
- Colab exercises: notebook experiments, Gemini model comparison, `gemini-embedding-001`, Gradio 6 chat UI.
- Review fixes: lab exercise cells turned back into tasks (reference answers in hidden ✅ Answer cells), `result_limiter.py` name as in the lab, `gemini-2.5-flash` removed from C2 (not available to new API users), Gradio cell no longer blocks with `debug=True`, chatbot banner says Gemini instead of IBM Granite.

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai chromadb gradio

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

> 💡 **How to use the ✅ Answer cells:** every exercise is followed by a hidden, collapsed cell titled *✅ Answer — …*. Try the exercise yourself first. To see the reference answer, double-click the cell (or *Show code*), tick **run_answer** and run it. While the box is unticked the cell raises a short *Answer not run* message on purpose, so **Runtime ▸ Run all** stops right before the first answer instead of spoiling it — after that, run the remaining cells one by one (or tick the boxes you want and use *Run after*). The piped `!printf … | python script.py` demo after each lab answer needs that answer's file.

## Project files
All files go into `food_rag/`, matching the original repo layout.

In [ ]:
import os
os.makedirs('food_rag', exist_ok=True)
%cd food_rag
!wget -q -O FoodDataSet.json "https://raw.githubusercontent.com/hengkuankian/learning_ibm_rag_agentic/main/Course3_Vector_Databases_for_RAG/Food_Recommendation_System_Using_Chroma_DB/FoodDataSet.json"
import json; print(len(json.load(open('FoodDataSet.json'))), 'food items')

### `gemini_ef.py` — Gemini embedding function for Chroma
A tiny `chromadb.EmbeddingFunction` over `GoogleGenerativeAIEmbeddings` (swap rule: *Raw ChromaDB → GeminiEmbeddingFunction*). The lab scripts import it instead of `SentenceTransformerEmbeddingFunction`; they read `GOOGLE_API_KEY` from the environment set in Cell 2.

In [ ]:
%%writefile gemini_ef.py
# Chroma embedding function backed by Gemini (replaces SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2"))
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
from chromadb import Documents, EmbeddingFunction, Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model: str = "models/gemini-embedding-2"):
        self.model = model
        self._emb = GoogleGenerativeAIEmbeddings(model=model)   # key read from GOOGLE_API_KEY

    def __call__(self, input: Documents) -> Embeddings:
        texts = list(input)
        out = []
        for i in range(0, len(texts), 100):              # free tier: <= 100 texts per call
            out += self._emb.embed_documents(texts[i:i + 100])
        return out

    @staticmethod
    def name() -> str:
        return "gemini"

    def get_config(self):
        return {"model": self.model}

    @staticmethod
    def build_from_config(config):
        return GeminiEmbeddingFunction(config.get("model", "models/gemini-embedding-2"))

### `shared_functions.py` — data loading, collection creation, (filtered) similarity search
Unchanged from the lab: `load_food_data`, `create_similarity_search_collection` (cosine, `all-MiniLM-L6-v2`), `populate_similarity_collection`, `perform_similarity_search`, `perform_filtered_similarity_search` (cuisine + max-calorie `where` filters).

In [ ]:
%%writefile shared_functions.py
import chromadb
from chromadb.utils import embedding_functions
from gemini_ef import GeminiEmbeddingFunction  # Colab: Gemini embeddings
import json
import re
import numpy as np
from typing import List, Dict, Any, Optional

# Initialize ChromaDB client
client = chromadb.Client()

def load_food_data(file_path: str) -> List[Dict]:
    """Load food data from JSON file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            food_data = json.load(file)
        
        # Ensure each item has required fields and normalize the structure
        for i, item in enumerate(food_data):
            # Normalize food_id to string
            if 'food_id' not in item:
                item['food_id'] = str(i + 1)
            else:
                item['food_id'] = str(item['food_id'])
            
            # Ensure required fields exist
            if 'food_ingredients' not in item:
                item['food_ingredients'] = []
            if 'food_description' not in item:
                item['food_description'] = ''
            if 'cuisine_type' not in item:
                item['cuisine_type'] = 'Unknown'
            if 'food_calories_per_serving' not in item:
                item['food_calories_per_serving'] = 0
            
            # Extract taste features from nested food_features if available
            if 'food_features' in item and isinstance(item['food_features'], dict):
                taste_features = []
                for key, value in item['food_features'].items():
                    if value:
                        taste_features.append(str(value))
                item['taste_profile'] = ', '.join(taste_features)
            else:
                item['taste_profile'] = ''
        
        print(f"Successfully loaded {len(food_data)} food items from {file_path}")
        return food_data
        
    except Exception as e:
        print(f"Error loading food data: {e}")
        return []

def create_similarity_search_collection(collection_name: str, collection_metadata: dict = None):
    """Create ChromaDB collection with sentence transformer embeddings"""
    try:
        # Try to delete existing collection to start fresh
        client.delete_collection(collection_name)
    except:
        pass
    
    # Create embedding function
    sentence_transformer_ef = GeminiEmbeddingFunction()
    
    # Create new collection
    return client.create_collection(
        name=collection_name,
        metadata=collection_metadata,
        configuration={
            "hnsw": {"space": "cosine"},
            "embedding_function": sentence_transformer_ef
        }
    )

def populate_similarity_collection(collection, food_items: List[Dict]):
    """Populate collection with food data and generate embeddings"""
    documents = []
    metadatas = []
    ids = []
    
    # Create unique IDs to avoid duplicates
    used_ids = set()
    
    for i, food in enumerate(food_items):
        # Create comprehensive text for embedding using rich JSON structure
        text = f"Name: {food['food_name']}. "
        text += f"Description: {food.get('food_description', '')}. "
        text += f"Ingredients: {', '.join(food.get('food_ingredients', []))}. "
        text += f"Cuisine: {food.get('cuisine_type', 'Unknown')}. "
        text += f"Cooking method: {food.get('cooking_method', '')}. "
        
        # Add taste profile from food_features
        taste_profile = food.get('taste_profile', '')
        if taste_profile:
            text += f"Taste and features: {taste_profile}. "
        
        # Add health benefits if available
        health_benefits = food.get('food_health_benefits', '')
        if health_benefits:
            text += f"Health benefits: {health_benefits}. "
        
        # Add nutritional information
        if 'food_nutritional_factors' in food:
            nutrition = food['food_nutritional_factors']
            if isinstance(nutrition, dict):
                nutrition_text = ', '.join([f"{k}: {v}" for k, v in nutrition.items()])
                text += f"Nutrition: {nutrition_text}."
        
        # Generate unique ID to avoid duplicates
        base_id = str(food.get('food_id', i))
        unique_id = base_id
        counter = 1
        while unique_id in used_ids:
            unique_id = f"{base_id}_{counter}"
            counter += 1
        used_ids.add(unique_id)
        
        documents.append(text)
        ids.append(unique_id)
        metadatas.append({
            "name": food["food_name"],
            "cuisine_type": food.get("cuisine_type", "Unknown"),
            "ingredients": ", ".join(food.get("food_ingredients", [])),
            "calories": food.get("food_calories_per_serving", 0),
            "description": food.get("food_description", ""),
            "cooking_method": food.get("cooking_method", ""),
            "health_benefits": food.get("food_health_benefits", ""),
            "taste_profile": food.get("taste_profile", "")
        })
    
    # Add all data to collection
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )
    
    print(f"Added {len(food_items)} food items to collection")

def perform_similarity_search(collection, query: str, n_results: int = 5) -> List[Dict]:
    """Perform similarity search and return formatted results"""
    try:
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
        
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            return []
        
        formatted_results = []
        for i in range(len(results['ids'][0])):
            # Calculate similarity score (1 - distance)
            similarity_score = 1 - results['distances'][0][i]
            
            result = {
                'food_id': results['ids'][0][i],
                'food_name': results['metadatas'][0][i]['name'],
                'food_description': results['metadatas'][0][i]['description'],
                'cuisine_type': results['metadatas'][0][i]['cuisine_type'],
                'food_calories_per_serving': results['metadatas'][0][i]['calories'],
                'similarity_score': similarity_score,
                'distance': results['distances'][0][i]
            }
            formatted_results.append(result)
        
        return formatted_results
        
    except Exception as e:
        print(f"Error in similarity search: {e}")
        return []

def perform_filtered_similarity_search(collection, query: str, cuisine_filter: str = None, 
                                     max_calories: int = None, n_results: int = 5) -> List[Dict]:
    """Perform filtered similarity search with metadata constraints"""
    where_clause = None
    
    # Build filters list
    filters = []
    if cuisine_filter:
        filters.append({"cuisine_type": cuisine_filter})
    
    if max_calories:
        filters.append({"calories": {"$lte": max_calories}})
    
    # Construct where clause based on number of filters
    if len(filters) == 1:
        where_clause = filters[0]
    elif len(filters) > 1:
        where_clause = {"$and": filters}
    
    try:
        results = collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where_clause
        )
        
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            return []
        
        formatted_results = []
        for i in range(len(results['ids'][0])):
            similarity_score = 1 - results['distances'][0][i]
            
            result = {
                'food_id': results['ids'][0][i],
                'food_name': results['metadatas'][0][i]['name'],
                'food_description': results['metadatas'][0][i]['description'],
                'cuisine_type': results['metadatas'][0][i]['cuisine_type'],
                'food_calories_per_serving': results['metadatas'][0][i]['calories'],
                'similarity_score': similarity_score,
                'distance': results['distances'][0][i]
            }
            formatted_results.append(result)
        
        return formatted_results
        
    except Exception as e:
        print(f"Error in filtered search: {e}")
        return []

## Step 1 — Interactive search CLI (`interactive_search.py`)

In [ ]:
%%writefile interactive_search.py
from shared_functions import *

# Global variable to store loaded food items
food_items = []

def main():
    """Main function for interactive CLI food recommendation system"""
    try:
        print("🍽️  Interactive Food Recommendation System")
        print("=" * 50)
        print("Loading food database...")
        
        # Load food data from file
        global food_items
        food_items = load_food_data('./FoodDataSet.json')
        print(f"✅ Loaded {len(food_items)} food items successfully")
        
        # Create and populate search collection
        collection = create_similarity_search_collection(
            "interactive_food_search",
            {'description': 'A collection for interactive food search'}
        )
        populate_similarity_collection(collection, food_items)
        
        # Start interactive chatbot
        interactive_food_chatbot(collection)
        
    except Exception as error:
        print(f"❌ Error initializing system: {error}")

def interactive_food_chatbot(collection):
    """Interactive CLI chatbot for food recommendations"""
    print("\n" + "="*50)
    print("🤖 INTERACTIVE FOOD SEARCH CHATBOT")
    print("="*50)
    print("Commands:")
    print("  • Type any food name or description to search")
    print("  • 'help' - Show available commands")
    print("  • 'quit' or 'exit' - Exit the system")
    print("  • Ctrl+C - Emergency exit")
    print("-" * 50)
    
    while True:
        try:
            # Get user input
            user_input = input("\n🔍 Search for food: ").strip()
            
            # Handle empty input
            if not user_input:
                print("   Please enter a search term or 'help' for commands")
                continue
            
            # Handle exit commands
            if user_input.lower() in ['quit', 'exit', 'q']:
                print("\n👋 Thank you for using the Food Recommendation System!")
                print("   Goodbye!")
                break
            
            # Handle help command
            elif user_input.lower() in ['help', 'h']:
                show_help_menu()
            
            # Handle food search
            else:
                handle_food_search(collection, user_input)
                
        except KeyboardInterrupt:
            print("\n\n👋 System interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error processing request: {e}")

def show_help_menu():
    """Display help information for users"""
    print("\n📖 HELP MENU")
    print("-" * 30)
    print("Search Examples:")
    print("  • 'chocolate dessert' - Find chocolate desserts")
    print("  • 'Italian food' - Find Italian cuisine")
    print("  • 'sweet treats' - Find sweet desserts")
    print("  • 'baked goods' - Find baked items")
    print("  • 'low calorie' - Find lower-calorie options")
    print("\nCommands:")
    print("  • 'help' - Show this help menu")
    print("  • 'quit' - Exit the system")

def handle_food_search(collection, query):
    """Handle food similarity search with enhanced display"""
    print(f"\n🔍 Searching for '{query}'...")
    print("   Please wait...")
    
    # Perform similarity search
    results = perform_similarity_search(collection, query, 5)
    
    if not results:
        print("❌ No matching foods found.")
        print("💡 Try different keywords like:")
        print("   • Cuisine types: 'Italian', 'Thai', 'Mexican'")
        print("   • Ingredients: 'chicken', 'vegetables', 'cheese'")
        print("   • Descriptors: 'spicy', 'sweet', 'healthy'")
        return
    
    # Display results with rich formatting
    print(f"\n✅ Found {len(results)} recommendations:")
    print("=" * 60)
    
    for i, result in enumerate(results, 1):
        # Calculate percentage score
        percentage_score = result['similarity_score'] * 100
        
        print(f"\n{i}. 🍽️  {result['food_name']}")
        print(f"   📊 Match Score: {percentage_score:.1f}%")
        print(f"   🏷️  Cuisine: {result['cuisine_type']}")
        print(f"   🔥 Calories: {result['food_calories_per_serving']} per serving")
        print(f"   📝 Description: {result['food_description']}")
        
        # Add visual separator
        if i < len(results):
            print("   " + "-" * 50)
    
    print("=" * 60)
    
    # Provide suggestions for further exploration
    suggest_related_searches(results)

def suggest_related_searches(results):
    """Suggest related searches based on current results"""
    if not results:
        return
    
    # Extract cuisine types from results
    cuisines = list(set([r['cuisine_type'] for r in results]))
    
    print("\n💡 Related searches you might like:")
    for cuisine in cuisines[:3]:  # Limit to 3 suggestions
        print(f"   • Try '{cuisine} dishes' for more {cuisine} options")
    
    # Suggest calorie-based searches
    avg_calories = sum([r['food_calories_per_serving'] for r in results]) / len(results)
    if avg_calories > 350:
        print("   • Try 'low calorie' for lighter options")
    else:
        print("   • Try 'hearty meal' for more substantial dishes")

if __name__ == "__main__":
    main()

Piped demo (non-interactive). For a real session run `%run interactive_search.py` and type in the input box; `quit` exits.

In [ ]:
!printf 'chocolate dessert\nspicy vegetarian\nhelp\nquit\n' | python interactive_search.py

## Step 2 — Advanced search with metadata filters (`advanced_search.py`)
Menu: basic search, cuisine filter, calorie filter, combined filters, demonstrations, help, exit.

In [ ]:
%%writefile advanced_search.py
from shared_functions import *

def main():
    """Main function for advanced search demonstrations"""
    try:
        print("🔬 Advanced Food Search System")
        print("=" * 50)
        print("Loading food database with advanced filtering capabilities...")
        
        # Load food data from JSON file
        food_items = load_food_data('./FoodDataSet.json')
        print(f"✅ Loaded {len(food_items)} food items successfully")
        
        # Create collection specifically for advanced search operations
        collection = create_similarity_search_collection(
            "advanced_food_search",
            {'description': 'A collection for advanced search demos'}
        )
        populate_similarity_collection(collection, food_items)
        
        # Start the interactive advanced search interface
        interactive_advanced_search(collection)
        
    except Exception as error:
        print(f"❌ Error initializing advanced search system: {error}")

def interactive_advanced_search(collection):
    """Interactive advanced search with filtering options"""
    print("\n" + "="*50)
    print("🔧 ADVANCED SEARCH WITH FILTERS")
    print("="*50)
    print("Search Options:")
    print("  1. Basic similarity search")
    print("  2. Cuisine-filtered search")  
    print("  3. Calorie-filtered search")
    print("  4. Combined filters search")
    print("  5. Demonstration mode")
    print("  6. Help")
    print("  7. Exit")
    print("-" * 50)
    
    while True:
        try:
            choice = input("\n📋 Select option (1-7): ").strip()
            
            if choice == '1':
                perform_basic_search(collection)
            elif choice == '2':
                perform_cuisine_filtered_search(collection)
            elif choice == '3':
                perform_calorie_filtered_search(collection)
            elif choice == '4':
                perform_combined_filtered_search(collection)
            elif choice == '5':
                run_search_demonstrations(collection)
            elif choice == '6':
                show_advanced_help()
            elif choice == '7':
                print("👋 Exiting Advanced Search System. Goodbye!")
                break
            else:
                print("❌ Invalid option. Please select 1-7.")
                
        except KeyboardInterrupt:
            print("\n\n👋 System interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

def perform_basic_search(collection):
    """Perform basic similarity search without filters"""
    print("\n🔍 BASIC SIMILARITY SEARCH")
    print("-" * 30)
    
    query = input("Enter search query: ").strip()
    if not query:
        print("❌ Please enter a search term")
        return
    
    print(f"\n🔍 Searching for '{query}'...")
    results = perform_similarity_search(collection, query, 5)
    
    display_search_results(results, "Basic Search Results")

def perform_cuisine_filtered_search(collection):
    """Perform cuisine-filtered similarity search"""
    print("\n🍽️ CUISINE-FILTERED SEARCH")
    print("-" * 30)
    
    # Show available cuisines from our dataset
    cuisines = ["Italian", "Thai", "Mexican", "Indian", "Japanese", "French", 
                "Mediterranean", "American", "Health Food", "Dessert"]
    print("Available cuisines:")
    for i, cuisine in enumerate(cuisines, 1):
        print(f"  {i}. {cuisine}")
    
    query = input("\nEnter search query: ").strip()
    cuisine_choice = input("Enter cuisine number (or cuisine name): ").strip()
    
    if not query:
        print("❌ Please enter a search term")
        return
    
    # Handle cuisine selection - accept both number and text input
    cuisine_filter = None
    if cuisine_choice.isdigit():
        idx = int(cuisine_choice) - 1
        if 0 <= idx < len(cuisines):
            cuisine_filter = cuisines[idx]
    else:
        cuisine_filter = cuisine_choice
    
    if not cuisine_filter:
        print("❌ Invalid cuisine selection")
        return
    
    print(f"\n🔍 Searching for '{query}' in {cuisine_filter} cuisine...")
    results = perform_filtered_similarity_search(
        collection, query, cuisine_filter=cuisine_filter, n_results=5
    )
    
    display_search_results(results, f"Cuisine-Filtered Results ({cuisine_filter})")

def perform_calorie_filtered_search(collection):
    """Perform calorie-filtered similarity search"""
    print("\n🔥 CALORIE-FILTERED SEARCH")
    print("-" * 30)
    
    query = input("Enter search query: ").strip()
    max_calories_input = input("Enter maximum calories (or press Enter for no limit): ").strip()
    
    if not query:
        print("❌ Please enter a search term")
        return
    
    max_calories = None
    if max_calories_input.isdigit():
        max_calories = int(max_calories_input)
    
    print(f"\n🔍 Searching for '{query}'" + 
          (f" with max {max_calories} calories..." if max_calories else "..."))
    
    results = perform_filtered_similarity_search(
        collection, query, max_calories=max_calories, n_results=5
    )
    
    calorie_text = f"under {max_calories} calories" if max_calories else "any calories"
    display_search_results(results, f"Calorie-Filtered Results ({calorie_text})")

def perform_combined_filtered_search(collection):
    """Perform search with multiple filters combined"""
    print("\n🎯 COMBINED FILTERS SEARCH")
    print("-" * 30)
    
    query = input("Enter search query: ").strip()
    cuisine = input("Enter cuisine type (optional): ").strip()
    max_calories_input = input("Enter maximum calories (optional): ").strip()
    
    if not query:
        print("❌ Please enter a search term")
        return
    
    cuisine_filter = cuisine if cuisine else None
    max_calories = int(max_calories_input) if max_calories_input.isdigit() else None
    
    # Build description of applied filters
    filter_description = []
    if cuisine_filter:
        filter_description.append(f"cuisine: {cuisine_filter}")
    if max_calories:
        filter_description.append(f"max calories: {max_calories}")
    
    filter_text = ", ".join(filter_description) if filter_description else "no filters"
    
    print(f"\n🔍 Searching for '{query}' with {filter_text}...")
    
    results = perform_filtered_similarity_search(
        collection, query, 
        cuisine_filter=cuisine_filter, 
        max_calories=max_calories, 
        n_results=5
    )
    
    display_search_results(results, f"Combined Filtered Results ({filter_text})")

def run_search_demonstrations(collection):
    """Run predetermined demonstrations of different search types"""
    print("\n📊 SEARCH DEMONSTRATIONS")
    print("=" * 40)
    
    demonstrations = [
        {
            "title": "Italian Cuisine Search",
            "query": "creamy pasta",
            "cuisine_filter": "Italian",
            "max_calories": None
        },
        {
            "title": "Low-Calorie Healthy Options",
            "query": "healthy meal",
            "cuisine_filter": None,
            "max_calories": 300
        },
        {
            "title": "Asian Light Dishes",
            "query": "light fresh meal",
            "cuisine_filter": "Japanese",
            "max_calories": 250
        }
    ]
    
    for i, demo in enumerate(demonstrations, 1):
        print(f"\n{i}. {demo['title']}")
        print(f"   Query: '{demo['query']}'")
        
        filters = []
        if demo['cuisine_filter']:
            filters.append(f"Cuisine: {demo['cuisine_filter']}")
        if demo['max_calories']:
            filters.append(f"Max Calories: {demo['max_calories']}")
        
        if filters:
            print(f"   Filters: {', '.join(filters)}")
        
        results = perform_filtered_similarity_search(
            collection,
            demo['query'],
            cuisine_filter=demo['cuisine_filter'],
            max_calories=demo['max_calories'],
            n_results=3
        )
        
        display_search_results(results, demo['title'], show_details=False)
        
        input("\n⏸️  Press Enter to continue to next demonstration...")

def display_search_results(results, title, show_details=True):
    """Display search results in a formatted way"""
    print(f"\n📋 {title}")
    print("=" * 50)
    
    if not results:
        print("❌ No matching results found")
        print("💡 Try adjusting your search terms or filters")
        return
    
    for i, result in enumerate(results, 1):
        score_percentage = result['similarity_score'] * 100
        
        if show_details:
            print(f"\n{i}. 🍽️  {result['food_name']}")
            print(f"   📊 Similarity Score: {score_percentage:.1f}%")
            print(f"   🏷️  Cuisine: {result['cuisine_type']}")
            print(f"   🔥 Calories: {result['food_calories_per_serving']}")
            print(f"   📝 Description: {result['food_description']}")
        else:
            print(f"   {i}. {result['food_name']} ({score_percentage:.1f}% match)")
    
    print("=" * 50)

def show_advanced_help():
    """Display help information for advanced search"""
    print("\n📖 ADVANCED SEARCH HELP")
    print("=" * 40)
    print("Search Types:")
    print("  1. Basic Search - Standard similarity search")
    print("  2. Cuisine Filter - Search within specific cuisine types")
    print("  3. Calorie Filter - Search for foods under calorie limits")
    print("  4. Combined Filters - Use multiple filters together")
    print("  5. Demonstrations - See predefined search examples")
    print("\nTips:")
    print("  • Use descriptive terms: 'creamy', 'spicy', 'light'")
    print("  • Combine ingredients: 'chicken vegetables'")
    print("  • Try cuisine names: 'Italian', 'Thai', 'Mexican'")
    print("  • Filter by calories for dietary goals")

if __name__ == "__main__":
    main()

In [ ]:
!printf '2\npasta\nItalian\n3\nlight salad\n400\n4\nhealthy\nJapanese\n500\n7\n' | python advanced_search.py

## Step 3 — RAG chatbot with an LLM (`enhanced_rag_chatbot.py`)
The only file with IBM code — watsonx `ModelInference` replaced by `ChatGoogleGenerativeAI(model="gemini-3.8-flash")` and `.generate(...)['results'][0]['generated_text']` by `.invoke(prompt).text` (see the lines at the top). Commands: any food question, `compare`, `help`, `quit`.

In [ ]:
%%writefile enhanced_rag_chatbot.py
from shared_functions import *
from typing import List, Dict, Any
# from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes   # Colab: replaced by Gemini
# from ibm_watsonx_ai.foundation_models import ModelInference
from langchain_google_genai import ChatGoogleGenerativeAI   # key read from GOOGLE_API_KEY
import json

# Global variables
food_items = []

# Google Gemini configuration (replaces IBM watsonx.ai credentials + ModelInference)
model_id = 'gemini-3.8-flash'   # was 'ibm/granite-3-3-8b-instruct'

# Initialize the LLM model
model = ChatGoogleGenerativeAI(model=model_id, temperature=0.2)

def main():
    """Main function for enhanced RAG chatbot system"""
    try:
        print("🤖 Enhanced RAG-Powered Food Recommendation Chatbot")
        print("   Powered by Google Gemini & ChromaDB")
        print("=" * 55)
        
        # Load food data
        global food_items
        food_items = load_food_data('./FoodDataSet.json')
        print(f"✅ Loaded {len(food_items)} food items")
        
        # Create collection for RAG system
        collection = create_similarity_search_collection(
            "enhanced_rag_food_chatbot",
            {'description': 'Enhanced RAG chatbot with Google Gemini integration'}
        )
        populate_similarity_collection(collection, food_items)
        print("✅ Vector database ready")
        
        # Test LLM connection
        print("🔗 Testing LLM connection...")
        test_response = model.invoke("Hello").text
        if test_response:
            print("✅ LLM connection established")
        else:
            print("❌ LLM connection failed")
            return
        
        # Start enhanced RAG chatbot
        enhanced_rag_food_chatbot(collection)
        
    except Exception as error:
        print(f"❌ Error: {error}")

def prepare_context_for_llm(query: str, search_results: List[Dict]) -> str:
    """Prepare structured context from search results for LLM"""
    if not search_results:
        return "No relevant food items found in the database."
    
    context_parts = []
    context_parts.append("Based on your query, here are the most relevant food options from our database:")
    context_parts.append("")
    
    for i, result in enumerate(search_results[:3], 1):
        food_context = []
        food_context.append(f"Option {i}: {result['food_name']}")
        food_context.append(f"  - Description: {result['food_description']}")
        food_context.append(f"  - Cuisine: {result['cuisine_type']}")
        food_context.append(f"  - Calories: {result['food_calories_per_serving']} per serving")
        
        if result.get('food_ingredients'):
            ingredients = result['food_ingredients']
            if isinstance(ingredients, list):
                food_context.append(f"  - Key ingredients: {', '.join(ingredients[:5])}")
            else:
                food_context.append(f"  - Key ingredients: {ingredients}")
        
        if result.get('food_health_benefits'):
            food_context.append(f"  - Health benefits: {result['food_health_benefits']}")
        
        if result.get('cooking_method'):
            food_context.append(f"  - Cooking method: {result['cooking_method']}")
        
        if result.get('taste_profile'):
            food_context.append(f"  - Taste profile: {result['taste_profile']}")
        
        food_context.append(f"  - Similarity score: {result['similarity_score']*100:.1f}%")
        food_context.append("")
        
        context_parts.extend(food_context)
    
    return "\n".join(context_parts)

def generate_llm_rag_response(query: str, search_results: List[Dict]) -> str:
    """Generate response using Google Gemini with retrieved context"""
    try:
        # Prepare context from search results
        context = prepare_context_for_llm(query, search_results)
        
        # Build the prompt for the LLM
        prompt = f'''You are a helpful food recommendation assistant. A user is asking for food recommendations, and I've retrieved relevant options from a food database.

User Query: "{query}"

Retrieved Food Information:
{context}

Please provide a helpful, short response that:
1. Acknowledges the user's request
2. Recommends 2-3 specific food items from the retrieved options
3. Explains why these recommendations match their request
4. Includes relevant details like cuisine type, calories, or health benefits
5. Uses a friendly, conversational tone
6. Keeps the response concise but informative

Response:'''

        # Generate response using IBM Granite
        generated_response = model.invoke(prompt).text
        
        # Extract the generated text
        if generated_response:
            response_text = generated_response
            
            # Clean up the response if needed
            response_text = response_text.strip()
            
            # If response is too short, provide a fallback
            if len(response_text) < 50:
                return generate_fallback_response(query, search_results)
            
            return response_text
        else:
            return generate_fallback_response(query, search_results)
            
    except Exception as e:
        print(f"❌ LLM Error: {e}")
        return generate_fallback_response(query, search_results)

def generate_fallback_response(query: str, search_results: List[Dict]) -> str:
    """Generate fallback response when LLM fails"""
    if not search_results:
        return "I couldn't find any food items matching your request. Try describing what you're in the mood for with different words!"
    
    top_result = search_results[0]
    response_parts = []
    
    response_parts.append(f"Based on your request for '{query}', I'd recommend {top_result['food_name']}.")
    response_parts.append(f"It's a {top_result['cuisine_type']} dish with {top_result['food_calories_per_serving']} calories per serving.")
    
    if len(search_results) > 1:
        second_choice = search_results[1]
        response_parts.append(f"Another great option would be {second_choice['food_name']}.")
    
    return " ".join(response_parts)

def enhanced_rag_food_chatbot(collection):
    """Enhanced RAG-powered conversational food chatbot with Google Gemini"""
    print("\n" + "="*70)
    print("🤖 ENHANCED RAG FOOD RECOMMENDATION CHATBOT")
    print("   Powered by Google Gemini")
    print("="*70)
    print("💬 Ask me about food recommendations using natural language!")
    print("\nExample queries:")
    print("  • 'I want something spicy and healthy for dinner'")
    print("  • 'What Italian dishes do you recommend under 400 calories?'")
    print("  • 'I'm craving comfort food for a cold evening'")
    print("  • 'Suggest some protein-rich breakfast options'")
    print("\nCommands:")
    print("  • 'help' - Show detailed help menu")
    print("  • 'compare' - Compare recommendations for two different queries")
    print("  • 'quit' - Exit the chatbot")
    print("-" * 70)
    
    conversation_history = []
    
    while True:
        try:
            user_input = input("\n👤 You: ").strip()
            
            if not user_input:
                print("🤖 Bot: Please tell me what kind of food you're looking for!")
                continue
            
            if user_input.lower() in ['quit', 'exit', 'q']:
                print("\n🤖 Bot: Thank you for using the Enhanced RAG Food Chatbot!")
                print("      Hope you found some delicious recommendations! 👋")
                break
            
            elif user_input.lower() in ['help', 'h']:
                show_enhanced_rag_help()
            
            elif user_input.lower() in ['compare']:
                handle_enhanced_comparison_mode(collection)
            
            else:
                # Process the food query with enhanced RAG
                handle_enhanced_rag_query(collection, user_input, conversation_history)
                conversation_history.append(user_input)
                
                # Keep conversation history manageable
                if len(conversation_history) > 5:
                    conversation_history = conversation_history[-3:]
                
        except KeyboardInterrupt:
            print("\n\n🤖 Bot: Goodbye! Hope you find something delicious! 👋")
            break
        except Exception as e:
            print(f"❌ Bot: Sorry, I encountered an error: {e}")

def handle_enhanced_rag_query(collection, query: str, conversation_history: List[str]):
    """Handle user query with enhanced RAG approach using Google Gemini"""
    print(f"\n🔍 Searching vector database for: '{query}'...")
    
    # Perform similarity search with more results for better context
    search_results = perform_similarity_search(collection, query, 3)
    
    if not search_results:
        print("🤖 Bot: I couldn't find any food items matching your request.")
        print("      Try describing what you're in the mood for with different words!")
        return
    
    print(f"✅ Found {len(search_results)} relevant matches")
    print("🧠 Generating AI-powered response...")
    
    # Generate enhanced RAG response using IBM Granite
    ai_response = generate_llm_rag_response(query, search_results)
    
    print(f"\n🤖 Bot: {ai_response}")
    
    # Show detailed results for reference
    print(f"\n📊 Search Results Details:")
    print("-" * 45)
    for i, result in enumerate(search_results[:3], 1):
        print(f"{i}. 🍽️  {result['food_name']}")
        print(f"   📍 {result['cuisine_type']} | 🔥 {result['food_calories_per_serving']} cal | 📈 {result['similarity_score']*100:.1f}% match")
        if i < 3:
            print()

def handle_enhanced_comparison_mode(collection):
    """Enhanced comparison between two food queries using LLM"""
    print("\n🔄 ENHANCED COMPARISON MODE")
    print("   Powered by AI Analysis")
    print("-" * 35)
    
    query1 = input("Enter first food query: ").strip()
    query2 = input("Enter second food query: ").strip()
    
    if not query1 or not query2:
        print("❌ Please enter both queries for comparison")
        return
    
    print(f"\n🔍 Analyzing '{query1}' vs '{query2}' with AI...")
    
    # Get results for both queries
    results1 = perform_similarity_search(collection, query1, 3)
    results2 = perform_similarity_search(collection, query2, 3)
    
    # Generate AI-powered comparison
    comparison_response = generate_llm_comparison(query1, query2, results1, results2)
    
    print(f"\n🤖 AI Analysis: {comparison_response}")
    
    # Show side-by-side results
    print(f"\n📊 DETAILED COMPARISON")
    print("=" * 60)
    print(f"{'Query 1: ' + query1[:20] + '...' if len(query1) > 20 else 'Query 1: ' + query1:<30} | {'Query 2: ' + query2[:20] + '...' if len(query2) > 20 else 'Query 2: ' + query2}")
    print("-" * 60)
    
    max_results = max(len(results1), len(results2))
    for i in range(min(max_results, 3)):
        left = f"{results1[i]['food_name']} ({results1[i]['similarity_score']*100:.0f}%)" if i < len(results1) else "---"
        right = f"{results2[i]['food_name']} ({results2[i]['similarity_score']*100:.0f}%)" if i < len(results2) else "---"
        print(f"{left[:30]:<30} | {right[:30]}")

def generate_llm_comparison(query1: str, query2: str, results1: List[Dict], results2: List[Dict]) -> str:
    """Generate AI-powered comparison between two queries"""
    try:
        context1 = prepare_context_for_llm(query1, results1[:3])
        context2 = prepare_context_for_llm(query2, results2[:3])
        
        comparison_prompt = f'''You are analyzing and comparing two different food preference queries. Please provide a thoughtful comparison.

Query 1: "{query1}"
Top Results for Query 1:
{context1}

Query 2: "{query2}"
Top Results for Query 2:
{context2}

Please provide a short comparison that:
1. Highlights the key differences between these two food preferences
2. Notes any similarities or overlaps
3. Explains which query might be better for different situations
4. Recommends the best option from each query
5. Keeps the analysis concise but insightful

Comparison:'''

        generated_response = model.invoke(comparison_prompt).text
        
        if generated_response:
            return generated_response.strip()
        else:
            return generate_simple_comparison(query1, query2, results1, results2)
            
    except Exception as e:
        return generate_simple_comparison(query1, query2, results1, results2)

def generate_simple_comparison(query1: str, query2: str, results1: List[Dict], results2: List[Dict]) -> str:
    """Simple comparison fallback"""
    if not results1 and not results2:
        return "No results found for either query."
    if not results1:
        return f"Found results for '{query2}' but none for '{query1}'."
    if not results2:
        return f"Found results for '{query1}' but none for '{query2}'."
    
    return f"For '{query1}', I recommend {results1[0]['food_name']}. For '{query2}', {results2[0]['food_name']} would be perfect."

def show_enhanced_rag_help():
    """Display help information for enhanced RAG chatbot"""
    print("\n📖 ENHANCED RAG CHATBOT HELP")
    print("=" * 45)
    print("🧠 This chatbot uses Google Gemini to understand your")
    print("   food preferences and provide intelligent recommendations.")
    print("\nHow to get the best recommendations:")
    print("  • Be specific: 'healthy Italian pasta under 350 calories'")
    print("  • Mention preferences: 'spicy comfort food for cold weather'")
    print("  • Include context: 'light breakfast for busy morning'")
    print("  • Ask about benefits: 'protein-rich foods for workout recovery'")
    print("\nSpecial features:")
    print("  • 🔍 Vector similarity search finds relevant foods")
    print("  • 🧠 AI analysis provides contextual explanations")
    print("  • 📊 Detailed nutritional and cuisine information")
    print("  • 🔄 Smart comparison between different preferences")
    print("\nCommands:")
    print("  • 'compare' - AI-powered comparison of two queries")
    print("  • 'help' - Show this help menu")
    print("  • 'quit' - Exit the chatbot")
    print("\nTips for better results:")
    print("  • Use natural language - talk like you would to a friend")
    print("  • Mention dietary restrictions or preferences")
    print("  • Include meal timing (breakfast, lunch, dinner)")
    print("  • Specify if you want healthy, comfort, or indulgent options")

if __name__ == "__main__":
    main()

In [ ]:
!printf 'I want something healthy and spicy for dinner\ncompare\nchocolate dessert\nfruit salad\nquit\n' | python enhanced_rag_chatbot.py

Interactive version (type questions in the input box):

In [ ]:
# %run enhanced_rag_chatbot.py

## Step 4 — Compare the three systems (`system_comparison.py`)

In [ ]:
%%writefile system_comparison.py
from shared_functions import *
import time

def main():
    """Compare all three search systems with the same query"""
    print("🔬 FOOD SEARCH SYSTEMS COMPARISON")
    print("=" * 50)
    
    # Load data once for all systems
    food_items = load_food_data('./FoodDataSet.json')
    
    # Create collections for each system
    interactive_collection = create_similarity_search_collection("comparison_interactive")
    advanced_collection = create_similarity_search_collection("comparison_advanced")
    rag_collection = create_similarity_search_collection("comparison_rag")
    
    # Populate all collections
    populate_similarity_collection(interactive_collection, food_items)
    populate_similarity_collection(advanced_collection, food_items)
    populate_similarity_collection(rag_collection, food_items)
    
    # Test query
    test_query = "chocolate dessert"
    
    print(f"\n🔍 Testing query: '{test_query}'")
    print("=" * 50)
    
    # System 1: Interactive Search Style
    print("\n1️⃣ INTERACTIVE SEARCH APPROACH:")
    print("-" * 30)
    start_time = time.time()
    interactive_results = perform_similarity_search(interactive_collection, test_query, 3)
    interactive_time = time.time() - start_time
    
    for i, result in enumerate(interactive_results, 1):
        print(f"{i}. {result['food_name']} ({result['similarity_score']*100:.1f}% match)")
        print(f"   {result['food_description']}")
    print(f"⏱️ Response time: {interactive_time:.3f} seconds")
    
    # System 2: Advanced Search Style
    print("\n2️⃣ ADVANCED SEARCH APPROACH:")
    print("-" * 30)
    start_time = time.time()
    
    # Show basic search
    basic_results = perform_similarity_search(advanced_collection, test_query, 3)
    print("📋 Basic results:")
    for i, result in enumerate(basic_results, 1):
        print(f"   {i}. {result['food_name']} - {result['cuisine_type']} ({result['food_calories_per_serving']} cal)")
    
    # Show filtered search
    spicy_results = perform_filtered_similarity_search(
        advanced_collection, test_query, cuisine_filter="Indian", n_results=2
    )
    print("🌶️ Filtered for Indian cuisine:")
    for i, result in enumerate(spicy_results, 1):
        print(f"   {i}. {result['food_name']} ({result['similarity_score']*100:.1f}% match)")
    
    advanced_time = time.time() - start_time
    print(f"⏱️ Response time: {advanced_time:.3f} seconds")
    
    # System 3: RAG Chatbot Style
    print("\n3️⃣ RAG CHATBOT APPROACH:")
    print("-" * 30)
    start_time = time.time()
    
    rag_results = perform_similarity_search(rag_collection, test_query, 3)
    
    # Generate RAG-style response
    rag_response = f"Perfect! I found some excellent chocolate dessert options for you. "
    rag_response += f"I'd highly recommend the {rag_results[0]['food_name']} - it's a {rag_results[0]['similarity_score']*100:.0f}% match "
    rag_response += f"and offers that sweet, rich flavor you're craving. "
    if rag_results[0]['cuisine_type'] == 'American':
        rag_response += "American desserts are perfect for chocolate lovers! "
    rag_response += f"At {rag_results[0]['food_calories_per_serving']} calories, it's a delightful treat. "
    rag_response += f"You might also enjoy {rag_results[1]['food_name']} as an alternative."
    
    print(f"🤖 Bot: {rag_response}")
    
    rag_time = time.time() - start_time
    print(f"⏱️ Response time: {rag_time:.3f} seconds")
    
    # Comparison Summary
    print("\n📊 SYSTEM COMPARISON SUMMARY:")
    print("=" * 50)
    print("Interactive Search:")
    print("  ✅ Fast and simple")
    print("  ✅ Direct results display")
    print("  ❌ Limited context")
    
    print("\nAdvanced Search:")
    print("  ✅ Powerful filtering options")
    print("  ✅ Multiple search modes")
    print("  ✅ Precise control")
    print("  ❌ Requires user to know filter options")
    
    print("\nRAG Chatbot:")
    print("  ✅ Natural language interaction")
    print("  ✅ Contextual explanations")
    print("  ✅ Conversational experience")
    print("  ❌ More complex implementation")
    
    print(f"\n⏱️ Performance Comparison:")
    print(f"  Interactive: {interactive_time:.3f}s")
    print(f"  Advanced: {advanced_time:.3f}s")
    print(f"  RAG Chatbot: {rag_time:.3f}s")

if __name__ == "__main__":
    main()

In [ ]:
!python system_comparison.py

## Lab exercises
### Exercise 1 — Enhance the Interactive Search System (search history)
Modify the interactive search system to add **search history tracking**: keep track of the user's previous searches and let them view the history with a **`history`** command.

*Hint:* add a global variable to store the history, and create new command handlers.

Colab: write your version to **`interactive_search_2.py`** (the lab modifies `interactive_search.py` in place; a separate file keeps Step 1 unchanged). Start from the Step 1 cell: `%%writefile interactive_search_2.py`, then test it with the piped demo below the answer.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 1 · search history
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
from pathlib import Path
Path('interactive_search_2.py').write_text(r'''from shared_functions import *

# Global variable to store loaded food items
food_items = []
search_history = []

def main():
    """Main function for interactive CLI food recommendation system"""
    try:
        print("🍽️  Interactive Food Recommendation System")
        print("=" * 50)
        print("Loading food database...")
        
        # Load food data from file
        global food_items
        global search_history
        food_items = load_food_data('./FoodDataSet.json')
        print(f"✅ Loaded {len(food_items)} food items successfully")
        
        # Create and populate search collection
        collection = create_similarity_search_collection(
            "interactive_food_search",
            {'description': 'A collection for interactive food search'}
        )
        populate_similarity_collection(collection, food_items)
        
        # Start interactive chatbot
        interactive_food_chatbot(collection)
        
    except Exception as error:
        print(f"❌ Error initializing system: {error}")

def interactive_food_chatbot(collection):
    """Interactive CLI chatbot for food recommendations"""
    print("\n" + "="*50)
    print("🤖 INTERACTIVE FOOD SEARCH CHATBOT")
    print("="*50)
    print("Commands:")
    print("  • Type any food name or description to search")
    print("  • 'help' - Show available commands")
    print("  • 'history' - Show your previous searches")
    print("  • 'quit' or 'exit' - Exit the system")
    print("  • Ctrl+C - Emergency exit")
    print("-" * 50)
    
    while True:
        try:
            # Get user input
            user_input = input("\n🔍 Search for food: ").strip()
            
            # Handle empty input
            if not user_input:
                print("   Please enter a search term or 'help' for commands")
                continue
            
            # Handle exit commands
            if user_input.lower() in ['quit', 'exit', 'q']:
                print("\n👋 Thank you for using the Food Recommendation System!")
                print("   Goodbye!")
                break
            
            # Handle help command
            elif user_input.lower() in ['help', 'h']:
                show_help_menu()
            
            elif user_input.lower() in ['history']:
                handle_history_command()
            
            # Handle food search
            else:
                handle_food_search(collection, user_input)
                
        except KeyboardInterrupt:
            print("\n\n👋 System interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error processing request: {e}")

def show_help_menu():
    """Display help information for users"""
    print("\n📖 HELP MENU")
    print("-" * 30)
    print("Search Examples:")
    print("  • 'chocolate dessert' - Find chocolate desserts")
    print("  • 'Italian food' - Find Italian cuisine")
    print("  • 'sweet treats' - Find sweet desserts")
    print("  • 'baked goods' - Find baked items")
    print("  • 'low calorie' - Find lower-calorie options")
    print("\nCommands:")
    print("  • 'help' - Show this help menu")
    print("  • 'history' - Show your search history")
    print("  • 'quit' - Exit the system")

def handle_food_search(collection, query):
    """Handle food similarity search with enhanced display"""
    search_history.append(query)
    print(f"\n🔍 Searching for '{query}'...")
    print("   Please wait...")
    
    # Perform similarity search
    results = perform_similarity_search(collection, query, 5)
    
    if not results:
        print("❌ No matching foods found.")
        print("💡 Try different keywords like:")
        print("   • Cuisine types: 'Italian', 'Thai', 'Mexican'")
        print("   • Ingredients: 'chicken', 'vegetables', 'cheese'")
        print("   • Descriptors: 'spicy', 'sweet', 'healthy'")
        return
    
    # Display results with rich formatting
    print(f"\n✅ Found {len(results)} recommendations:")
    print("=" * 60)
    
    for i, result in enumerate(results, 1):
        # Calculate percentage score
        percentage_score = result['similarity_score'] * 100
        
        print(f"\n{i}. 🍽️  {result['food_name']}")
        print(f"   📊 Match Score: {percentage_score:.1f}%")
        print(f"   🏷️  Cuisine: {result['cuisine_type']}")
        print(f"   🔥 Calories: {result['food_calories_per_serving']} per serving")
        print(f"   📝 Description: {result['food_description']}")
        
        # Add visual separator
        if i < len(results):
            print("   " + "-" * 50)
    
    print("=" * 60)
    
    # Provide suggestions for further exploration
    suggest_related_searches(results)

def suggest_related_searches(results):
    """Suggest related searches based on current results"""
    if not results:
        return
    
    # Extract cuisine types from results
    cuisines = list(set([r['cuisine_type'] for r in results]))
    
    print("\n💡 Related searches you might like:")
    for cuisine in cuisines[:3]:  # Limit to 3 suggestions
        print(f"   • Try '{cuisine} dishes' for more {cuisine} options")
    
    # Suggest calorie-based searches
    avg_calories = sum([r['food_calories_per_serving'] for r in results]) / len(results)
    if avg_calories > 350:
        print("   • Try 'low calorie' for lighter options")
    else:
        print("   • Try 'hearty meal' for more substantial dishes")

def handle_history_command():
    """Display user's search history"""
    if not search_history:
        print("📝 No search history available")
        return
    
    print("\n📝 Your Search History:")
    print("-" * 30)
    for i, search in enumerate(search_history[-10:], 1):  # Show last 10
        print(f"{i}. {search}")

if __name__ == "__main__":
    main()''')
print('wrote interactive_search_2.py')


In [ ]:
!printf 'pizza\ncurry\nhistory\nquit\n' | python interactive_search_2.py

### Exercise 2 — Build a Food Calorie Checker (`calorie_checker.py`)
Create an interactive tool in a new file called `calorie_checker.py` that helps users find foods within their calorie budget:

1. **Ask the user for their calorie budget** — get the maximum calories they want.
2. **Let them search for foods and show if they fit the budget** — use filtered search to check calorie limits.

**Goal:** a complete interactive program that repeatedly asks for searches and shows budget-friendly results. Test it with the piped demo below the answer (`500` budget, then `burger`, `salad`, `quit`).

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 2 · calorie checker
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
from pathlib import Path
Path('calorie_checker.py').write_text(r'''# Create a file called calorie_checker.py
from shared_functions import *

def calorie_checker():
    """Interactive calorie budget checker for foods"""
    print("🔥 FOOD CALORIE BUDGET CHECKER")
    print("=" * 35)
    
    # Setup data
    food_items = load_food_data('FoodDataSet.json')
    collection = create_similarity_search_collection("calorie_checker")
    populate_similarity_collection(collection, food_items)
    print("✅ Food database loaded!")
    
    # Get user's calorie budget
    while True:
        try:
            budget = int(input("\n💪 What's your calorie budget per meal? "))
            if budget > 0:
                break
            else:
                print("Please enter a positive number!")
        except ValueError:
            print("Please enter a valid number!")
    
    print(f"\n🎯 Your calorie budget: {budget} calories")
    print("Now search for foods to see if they fit your budget!")
    
    # Interactive search loop
    while True:
        print("\n" + "-" * 40)
        search_term = input("🔍 Search for a food (or 'quit' to exit): ").strip()
        
        if search_term.lower() == 'quit':
            print("👋 Thanks for using the Calorie Checker!")
            break
        
        if not search_term:
            print("Please enter a food to search for!")
            continue
        
        # Search for foods within budget
        budget_results = perform_filtered_similarity_search(
            collection, search_term, max_calories=budget, n_results=5
        )
        
        # Also get regular results to show what's over budget
        all_results = perform_similarity_search(collection, search_term, 5)
        
        print(f"\n📋 Results for '{search_term}':")
        
        if budget_results:
            print(f"✅ FITS YOUR BUDGET ({budget} cal limit):")
            for i, result in enumerate(budget_results, 1):
                calories = result['food_calories_per_serving']
                remaining = budget - calories
                print(f"  {i}. {result['food_name']}")
                print(f"     Calories: {calories} (🟢 {remaining} cal remaining)")
                print(f"     Cuisine: {result['cuisine_type']}")
        else:
            print(f"❌ No foods found within your {budget} calorie budget!")
        
        # Show over-budget options
        over_budget = []
        for result in all_results:
            if result['food_calories_per_serving'] > budget:
                over_budget.append(result)
        
        if over_budget:
            print(f"\n🚫 OVER BUDGET (but similar to your search):")
            for i, result in enumerate(over_budget[:3], 1):
                calories = result['food_calories_per_serving']
                excess = calories - budget
                print(f"  {i}. {result['food_name']}")
                print(f"     Calories: {calories} (🔴 {excess} cal over budget)")
        
        # Show budget summary
        if budget_results:
            avg_calories = sum(r['food_calories_per_serving'] for r in budget_results) / len(budget_results)
            print(f"\n📊 Budget-friendly options average: {avg_calories:.0f} calories")

if __name__ == "__main__":
    calorie_checker()''')
print('wrote calorie_checker.py')


In [ ]:
!printf '500\nburger\nsalad\nquit\n' | python calorie_checker.py

### Exercise 3 — Query Different Result Counts (`result_limiter.py`)
Practice controlling similarity search results by varying the number of results returned in a new file called `result_limiter.py`:

1. **Test different result limits** — use the `n_results` parameter to get different amounts of results.
2. **Compare result quality** — see how result quality changes with more or fewer results.

*Hint:* use `perform_similarity_search()` with different `n_results` values (1, 3, 5, 10). How do they change quality for *spicy chicken*?

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 3 · result limiter
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
from pathlib import Path
Path('result_limiter.py').write_text(r'''from shared_functions import *

def test_result_limits():
    """Test how different result limits affect search quality"""
    print("📊 SIMILARITY SEARCH RESULT LIMITER")
    print("=" * 45)
    
    # Setup
    food_items = load_food_data('FoodDataSet.json')
    collection = create_similarity_search_collection("result_test")
    populate_similarity_collection(collection, food_items)
    
    # Test query
    query = "spicy chicken"
    print(f"🔍 Testing query: '{query}'\n")
    
    # Test different result limits
    result_limits = [1, 3, 5, 10]
    
    for limit in result_limits:
        print(f"📋 Getting top {limit} result(s):")
        print("-" * 30)
        
        results = perform_similarity_search(collection, query, limit)
        
        if results:
            for i, result in enumerate(results, 1):
                score = result['similarity_score']
                print(f"  {i}. {result['food_name']}")
                print(f"     Score: {score:.3f}")
                print(f"     Cuisine: {result['cuisine_type']}")
                print(f"     Calories: {result['food_calories_per_serving']}")
                print()
            
            # Show quality analysis
            scores = [r['similarity_score'] for r in results]
            avg_score = sum(scores) / len(scores)
            print(f"  📈 Average score: {avg_score:.3f}")
            print(f"  🎯 Best score: {max(scores):.3f}")
            if len(scores) > 1:
                print(f"  📉 Worst score: {min(scores):.3f}")
        else:
            print("  No results found!")
        
        print("=" * 45)
    
    # Interactive test with user input
    print("\n🎮 INTERACTIVE MODE:")
    print("Test your own queries with different result limits!")
    
    while True:
        user_query = input("\nEnter search query (or press Enter to exit): ").strip()
        if not user_query:
            break
        
        try:
            limit_input = input("How many results? (1-20): ").strip()
            limit = int(limit_input) if limit_input.isdigit() and 1 <= int(limit_input) <= 20 else 5
        except:
            limit = 5
        
        print(f"\n🔍 Searching for '{user_query}' (limit: {limit})")
        results = perform_similarity_search(collection, user_query, limit)
        
        if results:
            print(f"Found {len(results)} results:")
            for i, result in enumerate(results, 1):
                print(f"  {i}. {result['food_name']} (Score: {result['similarity_score']:.3f})")
            
            # Show quality analysis
            scores = [r['similarity_score'] for r in results]
            avg_score = sum(scores) / len(scores)
            print()
            print(f"  📈 Average score: {avg_score:.3f}")
            print(f"  🎯 Best score: {max(scores):.3f}")
            if len(scores) > 1:
                print(f"  📉 Worst score: {min(scores):.3f}")
        else:
            print("No results found!")
    
    print("\n👋 Thanks for testing result limits!")

if __name__ == "__main__":
    test_result_limits()''')
print('wrote result_limiter.py')


In [ ]:
!printf 'grilled fish\n4\n\n' | python result_limiter.py


## 🧪 Colab exercises
### C1 · Use the project functions in the notebook
Import `shared_functions` and build the collection once, so you can experiment without re-running scripts.

In [ ]:
import sys; sys.path.insert(0, ".")
from shared_functions import *
food_items = load_food_data("FoodDataSet.json")
col = create_similarity_search_collection("nb_food")
populate_similarity_collection(col, food_items)
for r in perform_filtered_similarity_search(col, "comfort food", max_calories=450, n_results=5):
    print(r)

### C2 · Compare Gemini models
Run the RAG answer (`bot.generate_llm_rag_response`) with the free Gemini models **`gemini-3.8-flash`** and **`gemini-3.5-flash-lite`** and compare speed and grounding (does it only recommend dishes from the context?). Needs `col` from C1.

*Note:* `gemini-2.5-flash` is no longer offered to new API users (Google serves 2.5 models only to accounts that used them before), so it is not part of the comparison; catch errors per model so one 404/429 does not stop the loop.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — C2 · compare Gemini models
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
import time
import enhanced_rag_chatbot as bot          # importing does not start the CLI (main() is behind __name__ guard)
question = "vegetarian high protein lunch"
hits = perform_similarity_search(col, question, n_results=3)
names = [h["food_name"] for h in hits]
print("retrieved:", names)
for name in ["gemini-3.8-flash", "gemini-3.5-flash-lite"]:
    bot.model = ChatGoogleGenerativeAI(model=name)   # default temperature (Google's recommendation for Gemini 3)
    t = time.time()
    try:
        ans = bot.generate_llm_rag_response(question, hits)
    except Exception as e:                            # e.g. 429 RESOURCE_EXHAUSTED on the free tier
        print(f"--- {name}: {type(e).__name__}: {str(e)[:150]}"); continue
    grounded = [n for n in names if n.lower() in ans.lower()]
    print(f"--- {name} ({time.time()-t:.1f}s) · mentions {len(grounded)}/{len(names)} retrieved dishes: {grounded}")
    print(ans[:600])
bot.model = llm                                   # restore the Cell 2 model


### C3 · Another embedding model
Change the collection to use `models/gemini-embedding-001` and rerun Exercise 3. Does *spicy chicken* improve?

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — C3 · gemini-embedding-001 vs gemini-embedding-2
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
import chromadb
from gemini_ef import GeminiEmbeddingFunction
gcol = chromadb.Client().get_or_create_collection(
    "food_gemini_001",                                  # own name: C1's `col` stays alive
    configuration={"hnsw": {"space": "cosine"},
                   "embedding_function": GeminiEmbeddingFunction("models/gemini-embedding-001")})
if gcol.count() == 0:                                   # re-run safe: embed the 185 items only once
    populate_similarity_collection(gcol, food_items)
for label, c in [("embedding-2  ", col), ("embedding-001", gcol)]:
    res = perform_similarity_search(c, "spicy chicken", n_results=5)
    print(label, [(r["food_name"], round(r["similarity_score"], 3)) for r in res])


### C4 · Gradio front end (optional)
Wrap the RAG answer in a chat UI (`gr.ChatInterface`) — preview for course 02-style apps. Use `col` from C1 and `generate_llm_rag_response` from `enhanced_rag_chatbot`.

*Colab tip:* `launch(share=True)` returns immediately in Colab and shows the app inline + a public link; `debug=True` would block the cell (and Run all) until you stop it. Close the old app before re-running, otherwise every run opens a new port (7861, 7862, …).

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — C4 · Gradio chat UI
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
import gradio as gr
import enhanced_rag_chatbot as bot
def chat(msg, history):
    hits = perform_similarity_search(col, msg, n_results=3)
    return bot.generate_llm_rag_response(msg, hits)
try:
    demo.close()                     # re-run safe: stop the previous app (frees its port)
except NameError:
    pass
demo = gr.ChatInterface(chat, title="Food RAG bot")
demo.launch(share=True)              # no debug=True -> the cell finishes, the app keeps running
